In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import evaluate
import EIDA


model_name = "roberta-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device) # layer 12개, d_model 768차원
# GLUE의 SST-2는 영화 리뷰에서 가져온 문장으로 감정 분석(긍정: 1, 부정: 0)을 학습하는 데이터셋.
# 예시: {'sentence': "hide new secretions from the parental units", 'label': 0}
# label의 종류에 맞춰 num_labels=2 옵션으로 RoBERTa 모델을 생성하면, RoBERTa의 masked language modeling 사전학습 과정에서 쓰이던 모델 맨 끝의 pooler가 없어지고 그 자리에 랜덤으로 초기화된 파라미터인 classifier가 붙음. classifier는 두 계층(768x768, 768x2)으로 구성되어 있음.


max_length = 72 # 모든 데이터를 잘라내지 않고 다룰 수 있는 충분한 길이
batch_size = 16
dataset = load_dataset("glue", "sst2")
def preprocess_function(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )
tokenized_dataset = dataset.map(preprocess_function, batched=True)

train_dataset = tokenized_dataset['train']
dev_dataset = tokenized_dataset['validation']


metric = evaluate.load("glue", "sst2") # GLUE 벤치마크에서 SST-2 셋에 대한 점수는 accuracy.
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=1)
    return metric.compute(predictions=predictions, references=labels)

c:\Users\3un8i\anaconda3\envs\IDA\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [2]:
# 모델의 원본 파라미터 중에서 classifier의 두 번째 계층인 (가중치 768x2, 편향 2) 부분을 제외한 모든 파라미터를 학습 불가능한 상태로 고정하고, 추가할 어댑터 안에만 학습 가능한 파라미터를 두려고 함.
# 각 레이어 안의 W_Q, W_K, W_V, W_O, W_fc1, W_fc2들과 마지막 classifier의 첫 번째 계층, 이렇게 73개의 파라미터는 뒤에서 EIDA.Linear_with_adapter 타입으로 교체할 때 requires_grad=False 설정이 이루어짐. 그 73개에 해당하지 않는 학습가능한 파라미터인 embedding과 layer normalization은 지금 고정함.
for name, param in model.roberta.embeddings.named_parameters(): # 모델의 embedding 부분
    param.requires_grad = False
for layer in model.roberta.encoder.layer: # 모델의 transformer layer 12개
    layer.attention.output.LayerNorm.weight.requires_grad = False
    layer.attention.output.LayerNorm.bias.requires_grad = False
    layer.output.LayerNorm.weight.requires_grad = False
    layer.output.LayerNorm.bias.requires_grad = False


for _ in range(5): # 5에폭의 학습을 수행하기 위한 반복문
    train_dataset = train_dataset.shuffle() # train set 섞기
    input_ids = torch.tensor(train_dataset['input_ids'])
    attention_mask = torch.tensor(train_dataset['attention_mask'])
    label = torch.tensor(train_dataset['label'])
    
    # token representation 표본 추출
    sample_inputs, sample_delta_outputs = EIDA.forward_roberta(model, input_ids, attention_mask, label, begin=0, end=256, batch_size=batch_size, max_length=max_length, N=2)
    # train set의 67,349 개의 데이터 중 end-begin=256 개를 표본 추출에 활용 (vRAM 8GB의 RTX 4060로 모든 표본을 device='cuda'에 두고 작업 가능)
    # 각 hidden states(input token representation으로서 49군데, output token representation으로서 73군데)의 위치마다 한 시퀀스 안에서 N=2 개의 토큰을 표본으로 추출
    # sample_inputs: (end-start)*N = 512 개의 벡터가 담긴 list 49개가 담긴 list
    # sample_delta_outputs: (end-start)*N = 512 개의 벡터가 담긴 list 73개가 담긴 list

    plane_inputs=[]
    # input token representation의 표본을 추출한 49군데의 index:
    # 4*l+0: layer[l] 안에서 W_Q, W_K, W_V들의 공통 input인 hidden states가 위치하는 곳 (768차원) (l = 0, 1, ..., 11)
    # 4*l+1: layer[l] 안에서 W_O의 input인 hidden states가 위치하는 곳 (768차원)
    # 4*l+2: layer[l] 안에서 W_fc1의 input인 hidden states가 위치하는 곳 (768차원)
    # 4*l+3: layer[l] 안에서 W_fc2의 input인 hidden states가 위치하는 곳 (3072차원)
    # 4*12+0: classifier의 input인 hidden states가 위치하는 곳 (768차원)
    for i in range(4*12+1):
        plane_inputs.append(EIDA.PCA(sample_inputs[i], plane_dim=32)) # 입력되는 512개의 점으로 주성분분석을 수행해서, 768(또는 3072)차원 공간에서 token representation의 분포를 가장 잘 포착하는 32차원 평면을 추정하는 함수
    del sample_inputs
    # plane_inputs는 [32, 768] 또는 [32, 3072]의 shape을 가진 torch.tensor 49개의 list
    # 각 텐서는 token representation space에서 PCA를 통해 추산된 32차원 부분공간으로의 projection map을 의미하고, 벡터 32개가 서로 orthonormal하게 있음. 이 맵이 어댑터의 A를 구성함.

    plane_delta_outputs=[]
    # output token representation의 표본을 추출한 73군데의 index:
    # 6*l+0: layer[l] 안에서 W_Q의 output인 hidden states가 위치하는 곳 (768차원)
    # 6*l+1: layer[l] 안에서 W_K의 output인 hidden states가 위치하는 곳 (768차원)
    # 6*l+2: layer[l] 안에서 W_V의 output인 hidden states가 위치하는 곳 (768차원)
    # 6*l+3: layer[l] 안에서 W_O의 output인 hidden states가 위치하는 곳 (768차원)
    # 6*l+4: layer[l] 안에서 W_fc1의 output인 hidden states가 위치하는 곳 (3072차원)
    # 6*l+5: layer[l] 안에서 W_fc2의 output인 hidden states가 위치하는 곳 (768차원)
    # 6*12+0: classifier의 첫번째 계층(768x768)의 output인 hidden states가 위치하는 곳 (768차원)
    for i in range(6*12+1):
        plane_delta_outputs.append(EIDA.PCA(sample_delta_outputs[i], plane_dim=32))
    del sample_delta_outputs
    # plane_delta_outputs는 [32, 768] 또는 [32, 3072]의 shape을 가진 torch.tensor 73개의 list
    # 각 텐서는 token representation space에서 PCA를 통해 추산된 32차원 부분공간으로의 projection map을 의미하고, 벡터 32개가 서로 orthonormal하게 있음. 이 맵의 transpose가 어댑터의 C를 구성함.


    # 모델 안의 73개의 파라미터를 각각 EIDA.Linear_with_adapter 타입으로 교체하는 과정
    # EIDA.Linear_with_adapter는 (원본 파라미터) + (C @ B @ A) 로 구성됨. 이 중에서 B만 학습가능한 파라미터.
    # A는 파라미터의 input token representation space에서 32차원 부분공간으로의 projection map, B는 32x32 행렬,
    # C는 파라미터의 output token representation space에서 32차원 부분공간으로의 projection map을 transpose해서 얻어진, 32차원에서 768(또는 3072)차원으로 가는 map.(각 열의 orthonormality 때문에 transpose가 projection의 역과정이 됨)
    for l, layer in enumerate(model.roberta.encoder.layer):
        layer.attention.self.query = EIDA.Linear_with_adapter(original_param=layer.attention.self.query, A=plane_inputs[4*l+0], C=plane_delta_outputs[6*l+0])
        layer.attention.self.key = EIDA.Linear_with_adapter(original_param=layer.attention.self.key, A=plane_inputs[4*l+0], C=plane_delta_outputs[6*l+1])
        layer.attention.self.value = EIDA.Linear_with_adapter(original_param=layer.attention.self.value, A=plane_inputs[4*l+0], C=plane_delta_outputs[6*l+2])
        layer.attention.output.dense = EIDA.Linear_with_adapter(original_param=layer.attention.output.dense, A=plane_inputs[4*l+1], C=plane_delta_outputs[6*l+3])
        layer.intermediate.dense = EIDA.Linear_with_adapter(original_param=layer.intermediate.dense, A=plane_inputs[4*l+2], C=plane_delta_outputs[6*l+4])
        layer.output.dense = EIDA.Linear_with_adapter(original_param=layer.output.dense, A=plane_inputs[4*l+3], C=plane_delta_outputs[6*l+5])
    model.classifier.dense = EIDA.Linear_with_adapter(original_param=model.classifier.dense, A=plane_inputs[4*12+0], C=plane_delta_outputs[6*12+0])
    # 이제 학습 가능한 파라미터의 수: 73*(32*32)+(768*2+2) = 76,290

    training_args = TrainingArguments(
        output_dir=os.path.join("results", "sst2"),
        save_strategy="no",
        learning_rate=2e-4,
        warmup_ratio=0.1,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=1,
        weight_decay=0.1,
        logging_dir="./logs",
        logging_steps=700,
        eval_strategy="steps",
        eval_steps=700,
        bf16=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 학습된 어댑터를 모델에 merge하고 모델을 저장하기
    for l, layer in enumerate(model.roberta.encoder.layer):
        layer.attention.self.query = layer.attention.self.query.merge()
        layer.attention.self.key = layer.attention.self.key.merge()
        layer.attention.self.value = layer.attention.self.value.merge()
        layer.attention.output.dense = layer.attention.output.dense.merge()
        layer.intermediate.dense = layer.intermediate.dense.merge()
        layer.output.dense = layer.output.dense.merge()
    model.classifier.dense = model.classifier.dense.merge()

    model.save_pretrained(os.path.join("results", "sst2"))

C:\Users\3un8i\AppData\Local\Temp\ipykernel_7532\191039685.py:84: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
 17%|█▋        | 700/4210 [00:41<03:19, 17.60it/s]

{'loss': 0.4793, 'grad_norm': 16.706817626953125, 'learning_rate': 0.0001852731591448931, 'epoch': 0.17}


                                                  
 17%|█▋        | 703/4210 [00:42<11:28,  5.09it/s]

{'eval_loss': 0.2751292884349823, 'eval_accuracy': 0.911697247706422, 'eval_runtime': 1.3213, 'eval_samples_per_second': 659.956, 'eval_steps_per_second': 41.626, 'epoch': 0.17}


 33%|███▎      | 1400/4210 [01:22<02:33, 18.35it/s]

...

{'loss': 0.2247, 'grad_norm': 1.3069943189620972, 'learning_rate': 0.0001113750329902349, 'epoch': 0.5}



 50%|████▉     | 2103/4210 [02:06<07:09,  4.91it/s]

{'eval_loss': 0.23822525143623352, 'eval_accuracy': 0.9243119266055045, 'eval_runtime': 1.424, 'eval_samples_per_second': 612.377, 'eval_steps_per_second': 38.625, 'epoch': 0.5}


 57%|█████▋    | 2392/4210 [02:22<01:44, 17.34it/s]

KeyboardInterrupt: 